<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-02-grafos/11%20-%20Modelagem%20da%20Tubulacao%20e%20Instrumentos%20como%20Grafo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 11 — Modelagem da tubulação e dos instrumentos como grafo

**ECAA08 · Grupo 06 · Arthur, Caique, Luis Felipe e Marina**  
**Planta:** drone agrícola de pulverização e estação de solo.


## Objetivo e continuidade da Etapa 1

Representar as conexões pelas quais a calda pode circular, preservando `EST_PMP_01`, `EST_XV_01`, `DRN_PMP_01` e a família `DRN_XV_01`. A rede tem um tanque de solo, uma bomba de recirculação/transferência, um reservatório embarcado, um filtro, uma bomba de pulverização e quatro seções de bicos. As quatro seções retomam o exemplo da Aula 06.

`EST_TQ_01`, `DRN_TQ_01`, os distribuidores e os filtros são identificadores de equipamentos introduzidos nesta etapa; não substituem as tags dos sensores. `EST_LT_01` mede o tanque de solo; `DRN_LT_01` mede o reservatório; `DRN_PT_01` e `DRN_FT_01` monitoram a descarga da bomba. Instrumentos que não transportam calda são atributos, não vértices hidráulicos em série.


## Modelo matemático

O dígrafo ponderado é $G=(V,E,w)$, com $w:E\to\mathbb{R}_{\geq0}$. Um vértice representa um equipamento ou uma seção; um arco representa uma ligação física orientada. O peso mede **comprimento em metros**, sem representar automaticamente perda de carga, tempo ou energia.

O catálogo básico tem $|V|=11$ e $|E|=11$. O lema dos graus determina $\sum_v d^+(v)=\sum_v d^-(v)=11$. A rede física contém o ciclo de recirculação `EST_TQ_01 → EST_PMP_01 → EST_MAN_01 → EST_TQ_01`. Uma rota alternativa pode existir mesmo em um dígrafo sem ciclos: dois ramos podem convergir adiante.

Passeio pode repetir arestas; trilha não repete arestas; caminho simples não repete vértices. O caminho da calda para um bico não representa a trajetória aérea do drone.


## Modos de operação e hipóteses

| Modo | Arestas utilizáveis | Interpretação |
|---|---|---|
| Mistura | e01, e02, e03 | Recirculação no tanque de solo |
| Abastecimento | e01, e02, e04 | Drone no solo, conexão de abastecimento acoplada |
| Voo | e05 a e11 | Abastecimento desconectado; pulverização embarcada |

Consultar o catálogo completo (`modo=None`) não autoriza abrir todas as válvulas ao mesmo tempo. Disponibilidade topológica é diferente do comando do atuador. A seleção de modos é uma hipótese explícita de arranjo hidráulico; comprimentos e diâmetros são dados de simulação.

O segundo filtro `DRN_FLT_02`, e12/e13 e válvulas de isolamento são uma **extensão proposta** para investigar redundância. O modelo básico funciona sem essa extensão. O ramo alternativo mantém a filtragem; não desvia calda por fora do filtro.

![Rede hidráulica](attachment:Grafo_Hidraulico.png)


## Representação e exemplo

A classe `GrafoTubulacao` usa listas de adjacência e catálogo de arestas com ID, peso, válvula, diâmetro, modo e disponibilidade. Rejeita extremidades inexistentes, pesos negativos, IDs duplicados e arestas paralelas, em vez de perder informações silenciosamente. A inspeção da Aula 16 usa um multigrafo separado.

`DRN_MAN_01` tem uma entrada e quatro saídas. Isso representa distribuição entre seções; não demonstra quatro caminhos independentes desde o tanque, pois todos compartilham a bomba. As matrizes são exportadas sob demanda para manter uma única fonte de dados.


## Atividades resolvidas e verificações

1. Some os graus: ambos os totais devem ser 11 no catálogo básico.
2. Em voo, e04 deve desaparecer da adjacência operacional. O abastecimento não acompanha o drone.
3. Ative a extensão: a ordem passa a 12 e o tamanho a 13.
4. Diferencie o sensor `DRN_LT_01` do equipamento `DRN_TQ_01`: o primeiro mede; o segundo contém calda.

**Entregável:** classe de modelagem, cadastro, graus e adjacência operacional, executados no notebook correspondente.


## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [1]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [2]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [3]:
def matriz_texto(M, linhas, colunas):
    return tabela([{'no': r, **{c: ('inf' if M[i][j] == inf else round(M[i][j], 3))
                   for j, c in enumerate(colunas)}} for i, r in enumerate(linhas)])


In [4]:
class GrafoTubulacao:
    """Dígrafo simples; catálogo físico e disponibilidade operacional separados.

    Sensores são atributos; válvulas podem estar nas arestas. Não há suposição
    de uma válvula em cada trecho. Arestas paralelas são rejeitadas.
    """
    def __init__(self, vertices):
        self.vertices = list(vertices)
        if len(set(self.vertices)) != len(self.vertices):
            raise ValueError('Vértices duplicados')
        self.arestas = {}
        self.adj = {v: [] for v in self.vertices}

    def adicionar_tubulacao(self, id_, origem, destino, comprimento_m,
                           valvula=None, diametro_mm=8.0, modos=('voo',)):
        if origem not in self.adj or destino not in self.adj:
            raise KeyError('Extremidade não cadastrada')
        if origem == destino or id_ in self.arestas:
            raise ValueError('Laço ou ID repetido')
        if any(self.arestas[e]['destino'] == destino for e in self.adj[origem]):
            raise ValueError('Aresta paralela: use um multigrafo com IDs próprios')
        if not isfinite(comprimento_m) or comprimento_m < 0 or not isfinite(diametro_mm) or diametro_mm <= 0:
            raise ValueError('Peso finito não negativo e diâmetro positivo são exigidos')
        self.arestas[id_] = dict(origem=origem, destino=destino, peso=float(comprimento_m),
                                 valvula=valvula, diametro_mm=diametro_mm,
                                 modos=tuple(modos), disponivel=True)
        self.adj[origem].append(id_)

    def vizinhos(self, u, modo=None, bloqueados=()):
        if u not in self.adj:
            raise KeyError(u)
        if u in bloqueados:
            return
        for id_ in self.adj[u]:
            a = self.arestas[id_]
            if a['disponivel'] and (modo is None or modo in a['modos']) and a['destino'] not in bloqueados:
                yield a['destino'], a['peso'], id_

    def bloquear(self, id_):
        self.arestas[id_]['disponivel'] = False

    def matrizes(self, modo=None):
        n = len(self.vertices)
        idx = {v: i for i, v in enumerate(self.vertices)}
        ids = list(self.arestas)
        A = [[0]*n for _ in range(n)]
        W = [[0.0 if i == j else inf for j in range(n)] for i in range(n)]
        B = [[0]*len(ids) for _ in range(n)]
        for j, id_ in enumerate(ids):
            a = self.arestas[id_]; u, v = idx[a['origem']], idx[a['destino']]
            # B descreve o catálogo físico, inclusive trechos indisponíveis.
            B[u][j], B[v][j] = -1, 1
            if a['disponivel'] and (modo is None or modo in a['modos']):
                A[u][v], W[u][v] = 1, a['peso']
        return A, W, B, ids

    def graus(self, modo=None):
        A, _, _, _ = self.matrizes(modo)
        return [{'no': v, 'entrada': sum(r[i] for r in A), 'saida': sum(A[i])}
                for i, v in enumerate(self.vertices)]


In [5]:
def criar_rede(redundante=False):
    nos = ['EST_TQ_01', 'EST_PMP_01', 'EST_MAN_01', 'DRN_TQ_01',
           'DRN_FLT_01', 'DRN_PMP_01', 'DRN_MAN_01'] + ['DRN_BICO_'+s for s in 'ABCD']
    if redundante:
        nos.append('DRN_FLT_02')
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('e01', nos[0], nos[1], 1.5, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e02', nos[1], nos[2], 1.0, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e03', nos[2], nos[0], 2.0, 'EST_XV_REC', 20, ('mistura',))
    g.adicionar_tubulacao('e04', nos[2], nos[3], 3.0, 'EST_XV_01', 15, ('abastecimento',))
    g.adicionar_tubulacao('e05', nos[3], nos[4], 0.4, 'DRN_XV_F1_IN')
    g.adicionar_tubulacao('e06', nos[4], nos[5], 0.3, 'DRN_XV_F1_OUT')
    g.adicionar_tubulacao('e07', nos[5], nos[6], 0.5)
    for i, s in enumerate('ABCD'):
        g.adicionar_tubulacao(f'e{8+i:02}', nos[6], 'DRN_BICO_'+s, 0.6+0.2*i, 'DRN_XV_01'+s)
    if redundante:
        g.adicionar_tubulacao('e12', 'DRN_TQ_01', 'DRN_FLT_02', 0.5, 'DRN_XV_F2_IN')
        g.adicionar_tubulacao('e13', 'DRN_FLT_02', 'DRN_PMP_01', 0.6, 'DRN_XV_F2_OUT')
    return g


## Experimento 1 — Cadastro e graus

In [6]:
rede = criar_rede()
print(tabela([{'id': k, **a} for k,a in rede.arestas.items()]))
print('\nGRAUS DO CATÁLOGO')
print(tabela(rede.graus()))
assert len(rede.vertices) == 11 and len(rede.arestas) == 11
assert sum(x['entrada'] for x in rede.graus()) == 11
assert sum(x['saida'] for x in rede.graus()) == 11
assert next(x for x in rede.graus() if x['no']=='DRN_MAN_01')['saida'] == 4


id  | origem     | destino    | peso               | valvula       | diametro_mm | modos                        | disponivel
----+------------+------------+--------------------+---------------+-------------+------------------------------+-----------
e01 | EST_TQ_01  | EST_PMP_01 | 1.5                | None          | 20          | ('mistura', 'abastecimento') | True      
e02 | EST_PMP_01 | EST_MAN_01 | 1.0                | None          | 20          | ('mistura', 'abastecimento') | True      
e03 | EST_MAN_01 | EST_TQ_01  | 2.0                | EST_XV_REC    | 20          | ('mistura',)                 | True      
e04 | EST_MAN_01 | DRN_TQ_01  | 3.0                | EST_XV_01     | 15          | ('abastecimento',)           | True      
e05 | DRN_TQ_01  | DRN_FLT_01 | 0.4                | DRN_XV_F1_IN  | 8.0         | ('voo',)                     | True      
e06 | DRN_FLT_01 | DRN_PMP_01 | 0.3                | DRN_XV_F1_OUT | 8.0         | ('voo',)                     | True      


## Experimento 2 — Modos e validação

In [7]:
A, W, B, ids = rede.matrizes('voo')
print(matriz_texto(A, rede.vertices, rede.vertices))
i = {v:j for j,v in enumerate(rede.vertices)}
assert A[i['EST_MAN_01']][i['DRN_TQ_01']] == 0
assert W[i['EST_MAN_01']][i['DRN_TQ_01']] == inf
reserva = criar_rede(True)
assert (len(reserva.vertices),len(reserva.arestas)) == (12,13)
try:
    rede.adicionar_tubulacao('duplicata','DRN_TQ_01','DRN_FLT_01',0.4)
except ValueError:
    print('Aresta paralela rejeitada corretamente.')
else:
    raise AssertionError('Aresta paralela deveria ser rejeitada')
print('Rede básica 11/11; extensão 12/13; modo voo verificado.')


no         | EST_TQ_01 | EST_PMP_01 | EST_MAN_01 | DRN_TQ_01 | DRN_FLT_01 | DRN_PMP_01 | DRN_MAN_01 | DRN_BICO_A | DRN_BICO_B | DRN_BICO_C | DRN_BICO_D
-----------+-----------+------------+------------+-----------+------------+------------+------------+------------+------------+------------+-----------
EST_TQ_01  | 0         | 0          | 0          | 0         | 0          | 0          | 0          | 0          | 0          | 0          | 0         
EST_PMP_01 | 0         | 0          | 0          | 0         | 0          | 0          | 0          | 0          | 0          | 0          | 0         
EST_MAN_01 | 0         | 0          | 0          | 0         | 0          | 0          | 0          | 0          | 0          | 0          | 0         
DRN_TQ_01  | 0         | 0          | 0          | 0         | 1          | 0          | 0          | 0          | 0          | 0          | 0         
DRN_FLT_01 | 0         | 0          | 0          | 0         | 0          | 1          |